
# <span style="color:blue">Topic 1: Consumer Group</span>

## Definition

A **Consumer Group** is a collection of consumers that work together to consume data from a Kafka topic.

**Why do we need Consumer Groups?**

Suppose we have:

```text
Orders Topic
```

and only one consumer:

```text
Orders Topic
      ↓
      C1
```

If millions of messages arrive every minute, a single consumer may not be able to process them efficiently.

To scale message processing, Kafka introduces **Consumer Groups**.

---

## Example

Topic:

```text
Orders
```

Partitions:

```text
P0
P1
P2
```

Consumer Group:

```text
Group-A

C1
C2
C3
```

Kafka assigns:

```text
P0 → C1
P1 → C2
P2 → C3
```

This enables **parallel processing**.

---

## Golden Rule

Within the same Consumer Group:

```text
1 Partition = 1 Consumer
```

A partition can be assigned to only one consumer within a group.

### Invalid Example

```text
P0 → C1
P0 → C2
```

Reason:

```text
Duplicate processing
```

---

## Consumer Count vs Partition Count

### Case 1: Partitions = Consumers

```text
Partitions = 3
Consumers = 3
```

Assignment:

```text
P0 → C1
P1 → C2
P2 → C3
```

Best scenario.

---

### Case 2: Partitions > Consumers

```text
Partitions = 6
Consumers = 3
```

Assignment:

```text
P0,P1 → C1

P2,P3 → C2

P4,P5 → C3
```

A consumer can consume from multiple partitions.

---

### Case 3: Consumers > Partitions

```text
Partitions = 3
Consumers = 5
```

Assignment:

```text
P0 → C1
P1 → C2
P2 → C3

C4 → Idle
C5 → Idle
```

Extra consumers remain idle.

---

## Interview Question

### Maximum Active Consumers?

Formula:

```text
Maximum Active Consumers = Number of Partitions
```

Example:

```text
Partitions = 5

Consumers = 10
```

Result:

```text
5 Active Consumers
5 Idle Consumers
```

---

## Multiple Consumer Groups

Topic:

```text
Orders
```

Consumer Groups:

```text
Analytics Group

Fraud Group

Notification Group
```

Architecture:

```text
Orders Topic
        ↓

 ┌──────┬────────┬──────────┐
 ▼      ▼        ▼

Analytics Fraud Notification
```

Each group receives the same message independently.

---

## Consumer Group Offset

Each Consumer Group maintains its own offset.

Example:

```text
Group-A → Offset 100

Group-B → Offset 50
```

Both groups track progress independently.

---

## Real World Example

Uber:

```text
ride_requests topic
```

Consumer Groups:

```text
Driver Matching

Pricing

Fraud Detection

Analytics
```

Single ride request event can be processed by all groups.

---

## Benefits of Consumer Groups

✅ Parallel Processing

✅ Scalability

✅ Load Balancing

✅ Fault Tolerance

✅ Better Resource Utilization

---

## Important Interview Questions

### What is a Consumer Group?

A collection of consumers working together to consume data from a topic.

### Can multiple consumers read the same partition in the same group?

❌ No

### Can one consumer read multiple partitions?

✅ Yes

### Can multiple consumer groups read the same topic?

✅ Yes

### What is the maximum number of active consumers?

✅ Equal to the number of partitions

---

## Quick Revision

```text
Topic
   ↓
Partitions
   ↓
Consumer Group
   ↓
Consumers
```

Rule:

```text
1 Partition = 1 Consumer
(within a consumer group)
```

In [0]:
print("Partition")

%md

# <span style="color:green">Topic 2: Partition Deep Dive</span>

## What is a Partition?

A Partition is a physical division of a Kafka topic.

Example:

```text
Topic: Orders
```

Without partition:

```text
Orders Topic

Order1
Order2
Order3
Order4
```

With partitions:

```text
Orders Topic

P0
P1
P2
```

Messages are distributed across partitions.

```text
P0 → O1 O4 O7

P1 → O2 O5 O8

P2 → O3 O6 O9
```

---

## Why Do We Need Partitions?

Suppose:

```text
10 Million Messages / Second
```

One queue cannot handle such traffic efficiently.

Solution:

```text
Split Data Across Partitions
```

---

## Benefits of Partitions

### 1. Scalability

```text
More Partitions
=
More Capacity
```

---

### 2. Parallel Processing

Without partitions:

```text
1 Consumer
```

With partitions:

```text
P0 → C1
P1 → C2
P2 → C3
```

Multiple consumers can process data simultaneously.

---

### 3. Load Distribution

Partitions can be distributed across brokers.

```text
P0 → Broker1

P1 → Broker2

P2 → Broker3
```

This reduces load on a single broker.

---

## Partition Assignment

Producer decides which partition receives the message.

Kafka supports multiple strategies.

---

## Round Robin Partitioning

When no key is provided:

```text
Msg1 → P0
Msg2 → P1
Msg3 → P2
Msg4 → P0
Msg5 → P1
```

Messages are distributed evenly.

---

## Key-Based Partitioning ⭐

Producer sends a key.

Example:

```text
CustomerID = 1001
```

Kafka calculates:

```text
hash(customerID)
```

Result:

```text
1001 → P1

1001 → P1

1001 → P1
```

Same key always goes to the same partition.

---

## Why Key-Based Partitioning?

To maintain ordering.

Example:

```text
Customer 1001

Order Created

Payment Success

Order Shipped

Order Delivered
```

All events go to the same partition.

Therefore order is preserved.

---

## Kafka Ordering Guarantee

### Interview Question

Does Kafka guarantee ordering?

Answer:

✅ Yes

But only within a partition.

---

### Example

Partition:

```text
P0

Order1
Order2
Order3
Order4
```

Order is preserved.

---

### Across Partitions

```text
P0 → Order1

P1 → Order2

P2 → Order3
```

Global order is not guaranteed.

---

## How to Maintain Ordering?

Use a partition key.

Example:

```text
Key = Customer_ID
```

Result:

```text
Same Customer
      ↓
Same Partition
      ↓
Ordering Preserved
```

---

## Partition Count vs Consumer Count

### Example

```text
Partitions = 1

Consumers = 10
```

Result:

```text
1 Active
9 Idle
```

---

### Better Configuration

```text
Partitions = 10

Consumers = 10
```

More parallelism.

---

## Important Note

More partitions are not always better.

Too many partitions cause:

```text
More Metadata

More Network Overhead

More Management Complexity
```

A balance is required.

---

## Real World Example

Uber ride events:

```text
ride_requests
```

Partition Key:

```text
driver_id

or

rider_id
```

Benefit:

```text
Same Rider Events
      ↓
Same Partition
      ↓
Ordering Maintained
```

---

## Benefits of Partitions

✅ Scalability

✅ Parallel Processing

✅ Load Balancing

✅ Better Throughput

✅ Distributed Storage

---

## Important Interview Questions

### What is a partition?

A physical subdivision of a topic used for scalability and parallel processing.

---

### Why are partitions needed?

✅ Scalability

✅ Parallel Processing

✅ Load Distribution

---

### Does Kafka guarantee ordering?

✅ Within a partition

❌ Across partitions

---

### How are messages assigned to partitions?

1. Round Robin
2. Key-Based Partitioning

---

### Why use a partition key?

To keep related messages in the same partition and maintain ordering.

---

### Can a partition exist on multiple brokers?

✅ Yes, through replication.

---

## Quick Revision

```text
Topic
   ↓
Partitions
   ↓
Parallel Processing
   ↓
High Throughput
```

Most Important Interview Statement:

```text
Kafka guarantees ordering only within a partition,
not across partitions.
```